<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-10-tuning-and-evaluation/lesson-10.4-evaluation/notebooks/GCP_Capstone_10.4_Evaluation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.4 Vertex AI Evaluation — The Second Judge on the Lane's Own Answers
**Netsetos GenAI Engineering — GCP Capstone** · Module 10 · rebuilt on the live lane, 9 September 2026

The gate has scored the lane since 4.8: nine thresholds over 65 rows, no model in the loop. This lesson adds the other judge. The deployed API's answers are collected, never generated, and Gemini reads them for groundedness and fulfilment through Vertex AI Evaluation; a rubric written on the contract scores what the catalogue cannot; the two judges are compared row by row; trajectories come from the chat service's real `tool_calls`; a candidate revision is judged pairwise against the live one; and every run lands in Experiments named by the commit.


## Setup
The kit, the roster member, and three values: rows sent to the judge, a candidate URL when there is one, the chat service's URL.


In [ ]:
!pip install -q google-cloud-aiplatform[evaluation]==2.1.0 pandas==2.3.3 requests==2.34.2 google-cloud-storage==3.13.1

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
ROWS          = 24     # golden rows sent to the judge here; make judge sends all 64
CANDIDATE_URL = ""     # https://candidate---documind-api-NUMBER.us-central1.run.app, after make candidate (10.1)
CHAT_URL      = f"https://documind-chat-{NUMBER}.{REGION}.run.app"   # the chat service (8.x): three brains, one retrieve()

print("kit:", KIT, "| API:", API_URL, "| datasets:", f"gs://{DATASETS}/sft/")


## Cell 1: The API, the chat service, the identities


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, CALLED THE WAY THE UI CALLS IT: one ID token per request, minted AS the roster member,
# audience = the API (7.3's hour-long fuse never arms). The kit mints it (documind_tools._id_token).
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). They
# land in Cloud Logging first (the sink copies them to BigQuery); this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

def gcs_text(uri: str) -> str:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_text()

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: make_trainset, judge, tune, run_eval
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules: cache_manager, router, breakers, cost
print("helpers: api(), usage_rows(), gcs_text(); the kit's evals/ and rag-api/ on sys.path")


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


## Cell 2: The lane's own answers, collected
Bring-your-own responses: what `make judge` collects, on the first rows.


In [ ]:
import judge
from run_eval import load_golden, normalise

# THE LANE'S OWN ANSWERS, COLLECTED - NEVER GENERATED. Bring-your-own responses: every answer the judge reads
# was produced by the deployed API - retrieval, reranker, contract, roster, the routed tier, the whole lane -
# so the score is a score of the lane, not of a prompt in a notebook. judge.collect() is what `make judge`
# runs: the first ROWS golden rows here, all 64 there, as the roster member, one ID token per request.
golden = load_golden()
TOKEN = id_token_as(MEMBER_SA, API_URL)
t0 = time.time()
rows = judge.collect(API_URL, golden, TOKEN, "eval@documind.in", limit=ROWS)
ok = [r for r in rows if r["status"] == 200]
print(f"{len(ok)}/{len(rows)} answers in {time.time() - t0:.0f}s | model {next((r.get('model') for r in ok if r.get('model')), '?')}")
# THE CONTEXT IS THE CITED CHUNK, IN FULL. The API returns each citation's quote - the clause, at most twenty-five
# words - and a groundedness judge handed only quotes marks every answer that says more than its quote as
# unsupported: the first live make judge scored the lane 0.25 grounded for exactly that (F39). The kit reads the
# cited chunks' text back from the store (Firestore chunks/{chunk_id}, what the indexer wrote); the quotes stay
# as the fallback, and the cell says which the judge got.
hit = judge.enrich_context(ok, PROJECT_ID)
print(f"context: {hit}/{sum(len(r['cited']) for r in ok)} cited chunks read in full from the store")
r0 = next(r for r in ok if r["answerable"])
print(f"\n{r0['id']} {r0['question']}")
print("  answer :", r0["response"][:160])
print("  context:", r0["context"][:160].replace("\n", " | "))
assert len(ok) >= ROWS * 0.9, "the API is failing on the golden rows: fix the lane before judging it"

svc = json.loads(subprocess.run(["gcloud", "run", "services", "describe", "documind-api", "--region", REGION,
                                 "--project", PROJECT_ID, "--format=json"], capture_output=True, text=True).stdout)
GIT_SHA = {e["name"]: e.get("value", "") for e in svc["spec"]["template"]["spec"]["containers"][0].get("env", [])}.get("GIT_SHA") or "dev"
print("\nAPI revision GIT_SHA:", GIT_SHA, "- the Experiments run is named after it")


## Cell 3: The gate's maths, on the rows we hold
Computation-based, no model in the loop, blind to paraphrase.


In [ ]:
from collections import Counter

# THE GATE'S MATHS, ON THE ROWS WE HOLD. Computation-based metrics have no model in the loop. run_eval.py's
# nine thresholds are exactly that: a normalised string match (sixty -> 60, per cent -> %), a citation
# count, a refusal flag - cheap, deterministic, and blind to paraphrase, which is why a second judge exists.
# ROUGE and BLEU are the same family and worse here: a refusal row has no reference to overlap with, and a
# correct "60 days" scores badly against "sixty (60) days' notice". The gate decides; the judge explains.
def gate_verdict(r: dict) -> str:
    text = normalise(r["response"])
    if not r["answerable"]:
        return "ok" if not r["answerable_said"] else "ANSWERED (should refuse)"
    if not r["answerable_said"]:
        return "REFUSED"
    if not r["cited"]:
        return "no citation"
    return "ok" if all(normalise(m) in text for m in r.get("must_contain", [])) else f"missing {r.get('must_contain')}"

verdicts = {r["id"]: gate_verdict(r) for r in ok}
print(Counter(v if v == "ok" else v.split(" ")[0] for v in verdicts.values()))
for rid, v in verdicts.items():
    if v != "ok":
        print(f"  {rid:6} {v}")


## Cell 4: The second judge
Groundedness and fulfilment, regional, a score and an explanation per row, in an Experiments run named by the commit.


In [ ]:
import vertexai
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

# THE SECOND JUDGE. Vertex AI Evaluation is REGIONAL (us-central1; not on global). The frame is judge.to_frame:
# prompt, the lane's response, the context it cited, must_contain as the reference, one instruction. Two
# pointwise metrics from the catalogue - GROUNDEDNESS (is the answer supported by the context it cites?) and
# the SDK's name for "did it answer what was asked" - FULFILLMENT in older SDKs, INSTRUCTION_FOLLOWING in
# 2.1.0; the kit's judge.pointwise_metrics() resolves it and the first live run stopped on the old name (F36).
# Each is a 1-5 score with an explanation, per row. The same call judge.evaluate() makes; inline here for the
# per-row table the next cells read.
df = judge.to_frame(ok)
RUN = f"api-{GIT_SHA}-colab-{time.strftime('%H%M')}"        # unique per run: Experiments refuses a name that exists (F38)
vertexai.init(project=PROJECT_ID, location=REGION, experiment="documind-eval")
NAMES, METRICS = judge.pointwise_metrics()
SECOND = NAMES[1].lower()                                   # the metric's key in the tables: <name>/score, <name>/mean
print("pointwise:", " + ".join(NAMES))
task = EvalTask(dataset=df, metrics=METRICS, experiment="documind-eval")
result = task.evaluate(experiment_run_name=RUN)
print({k: round(v, 3) for k, v in result.summary_metrics.items() if k.endswith("/mean") or k == "row_count"})
table = result.metrics_table
print("\nby shape (groundedness mean):")
print(table.groupby("shape")["groundedness/score"].mean().round(2).to_string())
low = table.sort_values("groundedness/score").head(1).iloc[0]
print(f"\nlowest: {low['id']} score {low['groundedness/score']} - {str(low['groundedness/explanation'])[:220]}")


## Cell 5: A rubric on the contract


In [ ]:
from vertexai.evaluation import PointwiseMetric, PointwiseMetricPromptTemplate

# A RUBRIC ON THE CONTRACT. The catalogue's metrics know nothing about DocuMind. This one does: it reads the
# contract's own rules - a quote that is in the context, an answer that carries the figure, a refusal when
# the context does not answer - as a 1-5 rubric. It runs on ten rows: a custom metric is a judge call per
# row, and the price of a rubric is the price of running it on every commit.
citation_fidelity = PointwiseMetric(
    metric="citation_fidelity",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={"Quote fidelity": "Every quote the response relies on appears, verbatim or nearly, in the context.",
                  "Figure fidelity": "Every number, date, amount or percentage in the response is stated in the context.",
                  "Refusal discipline": "If the context does not answer the prompt, the response says so and adds nothing."},
        rating_rubric={"5": "All quotes and figures are in the context; refuses exactly when it should.",
                       "4": "One quote paraphrased, all figures correct.",
                       "3": "One figure not traceable to the context.",
                       "2": "Answers a question the context does not answer, or invents a quote.",
                       "1": "Most of the response is not in the context."},
        input_variables=["prompt", "response", "context"]))
r2 = EvalTask(dataset=df.head(10), metrics=[citation_fidelity], experiment="documind-eval").evaluate(experiment_run_name=RUN + "-fidelity")
print({k: round(v, 3) for k, v in r2.summary_metrics.items() if k.endswith("/mean")})
worst = r2.metrics_table.sort_values("citation_fidelity/score").head(1).iloc[0]
print(f"lowest: {worst['id']} score {worst['citation_fidelity/score']} - {str(worst['citation_fidelity/explanation'])[:240]}")


## Cell 6: Two judges, row by row
Where they disagree is where the next fix is.


In [ ]:
# TWO JUDGES, ROW BY ROW. The gate's verdict (Cell 3) beside the judge's groundedness score (Cell 4). Where
# they agree, nothing to read. Where the gate says ok and the judge gives a 1 or 2, the answer carried the
# right words for a reason the context does not support - the kind of miss a string match cannot see.
# Where the gate says miss and the judge gives a 5, the answer was right and phrased differently - the kind
# of miss normalise() exists to shrink. Both lists are short, and both are where the next fix is.
t = table[["id", "shape", "groundedness/score", "groundedness/explanation"]].copy()
t["gate"] = t["id"].map(verdicts)
gate_ok, judge_ok = t["gate"] == "ok", t["groundedness/score"] >= 4
print(f"{len(t)} rows | gate ok {gate_ok.sum()} | judge >= 4 {judge_ok.sum()} | both {int((gate_ok & judge_ok).sum())}")
for label, mask in (("gate ok, judge <= 2 (right words, wrong reason)", gate_ok & (t["groundedness/score"] <= 2)),
                    ("gate miss, judge 5 (right answer, other words)", ~gate_ok & (t["groundedness/score"] == 5))):
    print(f"\n{label}: {int(mask.sum())}")
    for _, r in t[mask].head(3).iterrows():
        print(f"  {r['id']:6} gate={str(r['gate'])[:30]:30} judge={r['groundedness/score']}  {str(r['groundedness/explanation'])[:120]}")


## Cell 7: Trajectories are the chat service's tool_calls


In [ ]:
# TRAJECTORIES ARE THE CHAT SERVICE'S tool_calls. The three brains (8.7) answer through the one retrieve(),
# and /v1/chat returns the tools each turn called. The reference trajectory for a grounded answer is ONE
# retrieve, then the answer. judge.trajectories() sends a few ACME rows to each brain and computes 10.4's
# three matches on the real calls - exact, in order, any order - not on a table typed by hand.
TOKEN_CHAT = id_token_as(MEMBER_SA, CHAT_URL)
tr = judge.trajectories(CHAT_URL, golden, TOKEN_CHAT, rows=4)
print(f"{'brain':10} {'exact':>5} {'in-order':>8} {'any-order':>9} {'calls':>6}  errors")
for brain in judge.BRAINS:
    b = [x for x in tr if x["brain"] == brain]
    print(f"{brain:10} {sum(x['exact'] for x in b):>5} {sum(x['in_order'] for x in b):>8} {sum(x['any_order'] for x in b):>9} "
          f"{sum(x['calls'] for x in b) / max(len(b), 1):>6.1f}  {[x['error'] for x in b if x.get('error')] or ''}")
live_rows = [x for x in tr if not x.get("error")]
assert live_rows and all(x["any_order"] for x in live_rows), "a brain answered without calling retrieve: read 8.7's contract before judging anything"
print("\nevery answered turn reached retrieve; `exact` counts the brains that called it once and stopped")


## Cell 8: Pairwise, against a candidate revision


In [ ]:
# PAIRWISE, AGAINST A CANDIDATE REVISION. 10.1's A/B, judged: the same rows collected from the candidate URL
# (make candidate: no traffic, a tuned endpoint or another model behind it), the live revision as the
# baseline, and Gemini saying which answer is better, row by row. The Experiments run is named for both.
if CANDIDATE_URL:
    # The same token: its audience is the service's canonical URL, and the candidate is the same service behind a
    # tag URL. A token minted for the tag URL is refused with 401 on every row (F42).
    cand = judge.collect(CANDIDATE_URL, golden, TOKEN, "eval@documind.in", limit=ROWS)
    df2 = judge.to_frame([r for r in cand if r["status"] == 200], baseline=ok)     # the candidate is judged; live is the baseline
    s = judge.evaluate(df2, "documind-eval", f"api-{GIT_SHA}-vs-candidate-colab", pairwise=True, project=PROJECT_ID)
    for k, v in sorted(s.items()):
        if "win_rate" in k or k.endswith("/mean"):
            print(f"  {k:56} {v:.3f}")
    wr = s.get("pairwise_question_answering_quality/candidate_model_win_rate", 0.0)
    print("\n" + ("ship" if wr >= 0.55 else "do not ship"), f"- candidate win rate {wr:.0%} against the 55% bar (and the gate must be green first)")
else:
    print("CANDIDATE_URL is empty. make tune -> make candidate GENERATOR_MODEL=<endpoint> RAG_MODEL_BASE=<base> -> paste the URL above.")
    print("From Cloud Shell the same cell is: make judge API_B=<candidate url>")


## Cell 9: Experiments, by commit


In [ ]:
from google.cloud import aiplatform

# EXPERIMENTS, BY COMMIT. Every judge run lands in documind-eval named api-<GIT_SHA>; "did the tuned model
# help" and "did last week's prompt change cost groundedness" are two rows of this table, not two memories.
aiplatform.init(project=PROJECT_ID, location=REGION)
runs = aiplatform.ExperimentRun.list(experiment="documind-eval")
print(f"{'run':44} {'groundedness':>12} {SECOND[:11]:>11}")
for run in sorted(runs, key=lambda r: r.name)[-8:]:
    m = run.get_metrics()
    g, f = m.get("groundedness/mean"), m.get(f"{SECOND}/mean")
    print(f"{run.name:44} {g if g is None else round(g, 3)!s:>12} {f if f is None else round(f, 3)!s:>11}")
print("\nConsole: Vertex AI -> Model Development -> Experiments -> documind-eval")


## Cell 10: The three tiers, as the lane runs them


In [ ]:
# THE THREE TIERS, AS THE LANE RUNS THEM. Not a class with placeholders: the commands, the bar each one
# enforces, and what it costs. The gate runs on every commit and has no model in the loop; the judge runs on
# a candidate and on a daily sample; the numbers land in the same experiment.
JUDGE_USD_PER_ROW_METRIC = 0.0005      # a judge call per row per metric, at flash rates - re-verify on the pricing page
TIERS = [("dev, every commit", "make eval-live",
          "run_eval.py: nine thresholds over 65 rows, no model in the loop; red blocks the merge (4.8's CI job)", 0.0),
         ("pre-prod, a candidate", "make judge API_B=<candidate url>",
          "pairwise win rate >= 55% against the live revision; Experiments run api-<sha>-vs-candidate", 64 * 3 * JUDGE_USD_PER_ROW_METRIC),
         ("production, daily", "make judge JUDGE_ARGS='--rows 16'",
          "groundedness and fulfilment on a sample; alert when the mean drops a point from the previous run", 16 * 2 * JUDGE_USD_PER_ROW_METRIC)]
print(f"{'tier':24} {'command':38} {'usd/run':>8}  bar")
for tier, cmd, bar, usd in TIERS:
    print(f"{tier:24} {cmd:38} {usd:>8.3f}  {bar}")
print("\nthe gate decides; the judge explains. Where they disagree, read the row (Cell 6).")


## Where this goes
- **10.1** is where the candidate comes from; **10.5** and **10.6** are judged the same way, through the gate and this judge.
- **12.7** owns the CI job that runs the gate on every commit; the judge is the pre-prod and daily tier beside it.

## ✅ Lesson 10.4 complete
- ✅ The lane's answers collected, never generated; the gate's verdict per row
- ✅ Groundedness and fulfilment by Gemini, per row with explanations, in Experiments as api-<sha>
- ✅ A rubric on the contract; the two judges compared where they disagree
- ✅ Trajectories from real tool_calls across three brains; pairwise against a candidate
- ✅ Three tiers with their commands, bars and costs
